# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the `mlcroissant` library. All data entities are referenced by their `@id` as per the Croissant standard schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Identifier:', getattr(metadata, 'identifier', '(none)'))
print('Version:', getattr(metadata, 'version', '(none)'))

## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s.

**Note**: In Croissant, a *record set* corresponds to a logical table or file containing rows (records). Each *field* describes a column or property and has a unique `@id`.

In [ ]:
# List available record sets by @id and list their fields by @id
record_sets = dataset.list_record_sets()
print(f"Found {len(record_sets)} record sets in this dataset.")

for rs in record_sets:
    print(f'---\nRecord Set: {rs["@id"]}')
    print(f'  Name: {rs.get("name") or rs.get("@id")}')
    # List its fields and their @id
    fields = dataset.list_fields(record_set=rs["@id"])
    if fields:
        print('  Fields:')
        for field in fields:
            print(f'    - @id: {field["@id"]} (name: {field.get("name")})')
    else:
        print('  No fields found in this record set')

## 3. Data Extraction
We will now load data from each record set into pandas DataFrames for analysis.

Each record set will be referenced via its Croissant `@id`. Fields (columns) will also be referenced by their `@id`.

In [ ]:
# Gather all the record set @ids
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]
print('Record Set @ids:', record_set_ids)
dataframes = {}

for record_set_id in record_set_ids:
    print(f'Loading records for {record_set_id} ...')
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f'  Columns: {df.columns.tolist()}')
        print(f'  Number of records: {len(df)}')
    else:
        print('  No records loaded')

# For demonstration, select the first non-empty DataFrame for further steps
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

assert main_record_set_id, 'No non-empty record sets found.'print(f'\nMain record set selected for analysis: {main_record_set_id}')
print('Sample rows:')
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the data: filter records based on a numeric field, normalize it, and group by a categorical field—all using `@id`s as references.

*(Modify the code below for the specific field `@id`s revealed in the overview above. If field `@id`s aren't known or there is only one table, demo the procedure with example field names. Replace `<numeric_field_id>` and `<group_field_id>` accordingly!)*

In [ ]:
from IPython.display import display
# For demonstration, print all columns in main record set
main_df = dataframes[main_record_set_id]
print('Columns in main record set:', main_df.columns.tolist())

# Try to infer a numeric field and a group field for demo
import numpy as np
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    if np.issubdtype(main_df[col].dtype, np.number):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Try to coerce any field
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
            if main_df[col].notnull().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

# For group field, pick first column with <5 unique non-numeric values
for col in main_df.columns:
    if col != numeric_field_id and main_df[col].nunique() > 1 and main_df[col].nunique() < 10:
        group_field_id = col
        break

print(f'Numeric field selected: {numeric_field_id}')
print(f'Group field selected: {group_field_id}')

# Filter records with numeric_field > threshold
if numeric_field_id and numeric_field_id in main_df:
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the chosen group field and compute the mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df)
else:
    print('No suitable numeric field found for filtering and normalization.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare means across groups (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid', palette='muted')

if numeric_field_id and numeric_field_id in main_df:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

- We loaded the FAIR^2 dataset with `mlcroissant` and listed all record sets and fields by their `@id`.
- Data from the main record set was loaded into a pandas DataFrame using only `@id` references.
- We performed basic EDA: filtered records by a numeric field, normalized the values, and grouped by a categorical field.
- Visualizations showed the distribution of numeric values and group-wise comparisons.

For further analysis, consult the Croissant schema documentation for this dataset and always reference fields and sets by their unique identifiers for clarity and reproducibility.